# Purpose

This notebook is intended to be run under the conda environment `MFMC`.

This experiment is a numerical-stability diagnostic for the ridge coefficient `ε` in the trace-based FMCA objective. It uses one fixed subject-dependent split and is not intended to replace the official 5-fold cross-validation results.

The goal is to quantify how the ridge coefficient in Eq. 22 affects:

- numerical stability of the mini-batch autocorrelation matrices,
- robustness of the linear solves used by the trace objective, and
- downstream emotion-classification performance on a held-out split.


In [ ]:
import gc
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_SEED = 2025
BATCH_SIZE = 200
N_ITERS = 10000
EVAL_EVERY = 500
LR = 3e-4
COV_BETA = 0.5
BETAS = (0.9, 0.999)
EMBED_DIM = 128
EPS_LIST = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
CLASSIFIER_EVAL_BATCH_SIZE = 512
SPLIT_TEST_SIZE = 0.2
LOSS_DTYPE = torch.float64
MAX_REASONABLE_LOSS = 1e8
PROJECT_ROOT = Path('/home/zhengdeyang/TAFFC_MFMC/MFMC')
BASELINE_DIR = PROJECT_ROOT / 'DEAP' / 'MFMC_Fusion_MLP'
DATA_DIR = PROJECT_ROOT / 'DEAP' / 'Data_processed'


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


set_seed(RANDOM_SEED)
device = get_device()
print(f'Using device: {device}')
print(f'Baseline reference directory: {BASELINE_DIR}')
print(f'Data directory: {DATA_DIR}')


# Experimental Design

We inspect and follow the baseline implementation style in `MFMC/DEAP/MFMC_Fusion_MLP`, especially for the DEAP tensor shapes, the `Advanced1DCNN_channel` encoders, the pair-fusion MLP heads, the light MLP classifier, and the alternating unsupervised/supervised training loop.

For this diagnostic notebook we keep the baseline MFMC-T training logic, including the moving covariance estimators, and only expose the ridge coefficient `ε` that is injected into each mini-batch autocorrelation matrix before the tracked update. This keeps the supplementary sweep aligned with the main implementation while still answering the reviewer question about numerical stabilization.

We keep the batch size fixed at 200, matching the main MFMC experiments, so that the only varying factor is the ridge coefficient `ε`. To reduce computational cost, we use 10,000 iterations for this diagnostic analysis.

The split is a fixed 80/20 subject-dependent split: for each subject, windows are divided into train and test subsets with the same random seed, and stratification is applied whenever the per-subject class counts allow it. The train/test partition is therefore held constant across all tested `ε` values.

We evaluate the downstream classifier on the EEG encoder representation over the held-out 20% windows.


# Numerical Stabilization of Eq. 22

We follow the baseline notebook in estimating second-order statistics from the mini-batch without explicit centering, i.e. the matrices behave as autocorrelation-style estimates:

- `R_x = X^T X / B`
- `R_y = Y^T Y / B`
- `P_xy = X^T Y / B`

For the supplementary `ε` sweep, each mini-batch autocorrelation matrix is ridge-regularized before it is fed into the baseline moving-average covariance tracker:

\[
\widetilde{R} = R + \varepsilon \cdot \frac{\operatorname{trace}(R)}{K} I.
\]

Concretely,

\[
\widetilde{R}_x = R_x + \varepsilon \cdot \frac{\operatorname{trace}(R_x)}{K} I, \qquad
\widetilde{R}_y = R_y + \varepsilon \cdot \frac{\operatorname{trace}(R_y)}{K} I.
\]

The tracked covariances then follow the same bias-corrected exponential moving-average update as the baseline notebook. This means the supplementary experiment changes only the ridge strength, not the core MFMC-T training objective.

For diagnostics we still report condition numbers of the ridge-regularized batch matrices, and we temporarily promote those matrix computations to `float64` to make the stability checks more robust.


# Data Loading

The notebook prefers the processed DEAP files in `MFMC/DEAP/Data_processed` and automatically resolves the temperature / skin-temperature modality from either `temp_data.npy` or `skt_data.npy`. If any required file is missing, the notebook raises a clear error instead of failing silently.


In [ ]:
REQUIRED_FILE_CANDIDATES = {
    'eeg': ['eeg_data.npy'],
    'eog': ['eog_data.npy'],
    'skt': ['temp_data.npy', 'skt_data.npy'],
    'labels': ['emotion_labels.npy'],
    'subject': ['subject.npy'],
}


def resolve_required_files(data_dir: Path) -> dict:
    resolved = {}
    missing = []
    for key, candidates in REQUIRED_FILE_CANDIDATES.items():
        match = next((data_dir / name for name in candidates if (data_dir / name).exists()), None)
        if match is None:
            missing.append(f"{key}: {candidates}")
        else:
            resolved[key] = match
    if missing:
        raise FileNotFoundError(
            f"Required DEAP processed files were not found. Please check DATA_DIR = {data_dir}. Missing entries: {missing}"
        )
    return resolved


resolved_files = resolve_required_files(DATA_DIR)
for key, value in resolved_files.items():
    print(f'{key:>7s}: {value.name}')

subject_np = np.load(resolved_files['subject'])
labels_np = np.load(resolved_files['labels'])
eeg_np = np.load(resolved_files['eeg']).astype(np.float32)
eog_np = np.load(resolved_files['eog']).astype(np.float32)
skt_np = np.load(resolved_files['skt']).astype(np.float32)

assert len(eeg_np) == len(eog_np) == len(skt_np) == len(labels_np) == len(subject_np)

print('\nLoaded array shapes:')
print('eeg   ', eeg_np.shape, eeg_np.dtype)
print('eog   ', eog_np.shape, eog_np.dtype)
print('skt   ', skt_np.shape, skt_np.dtype, f"(from {resolved_files['skt'].name})")
print('labels', labels_np.shape, labels_np.dtype)
print('subject', subject_np.shape, subject_np.dtype)
print('\nUnique subjects:', np.unique(subject_np).tolist())
print('Class counts:', np.bincount(labels_np))


class MultiModalDEAPDataset(Dataset):
    def __init__(self, eeg, eog, skt, labels, subjects, indices):
        self.eeg = torch.from_numpy(eeg[indices]).float()
        self.eog = torch.from_numpy(eog[indices]).float()
        self.skt = torch.from_numpy(skt[indices]).float()
        self.labels = torch.from_numpy(labels[indices]).long()
        self.subjects = torch.from_numpy(subjects[indices]).long()

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return {
            'eeg': self.eeg[idx],
            'eog': self.eog[idx],
            'skt': self.skt[idx],
            'label': self.labels[idx],
            'subject': self.subjects[idx],
        }


def make_subject_dependent_split(labels, subjects, test_size=0.2, seed=2025):
    train_parts = []
    test_parts = []
    split_notes = []
    unique_subjects = np.unique(subjects)
    for subj in unique_subjects:
        subj_indices = np.where(subjects == subj)[0]
        subj_labels = labels[subj_indices]
        if len(subj_indices) < 2:
            train_parts.append(subj_indices)
            test_parts.append(np.array([], dtype=int))
            split_notes.append(
                f'Subject {int(subj)} has only {len(subj_indices)} sample(s); all windows were assigned to train.'
            )
            continue
        class_counts = np.bincount(subj_labels, minlength=int(labels.max()) + 1)
        usable_counts = class_counts[class_counts > 0]
        can_stratify = len(np.unique(subj_labels)) > 1 and np.all(usable_counts >= 2)
        split_kwargs = dict(test_size=test_size, random_state=seed, shuffle=True)
        if can_stratify:
            tr_idx, te_idx = train_test_split(subj_indices, stratify=subj_labels, **split_kwargs)
        else:
            tr_idx, te_idx = train_test_split(subj_indices, stratify=None, **split_kwargs)
            split_notes.append(
                f'Subject {int(subj)} used non-stratified split because per-class counts were too small: {class_counts.tolist()}'
            )
        train_parts.append(tr_idx)
        test_parts.append(te_idx)
    train_indices = np.sort(np.concatenate(train_parts))
    test_indices = np.sort(np.concatenate(test_parts))
    return train_indices, test_indices, split_notes


train_indices, test_indices, split_notes = make_subject_dependent_split(
    labels_np,
    subject_np,
    test_size=SPLIT_TEST_SIZE,
    seed=RANDOM_SEED,
)

assert set(train_indices).isdisjoint(set(test_indices))
assert len(train_indices) + len(test_indices) == len(labels_np)

train_dataset = MultiModalDEAPDataset(eeg_np, eog_np, skt_np, labels_np, subject_np, train_indices)
test_dataset = MultiModalDEAPDataset(eeg_np, eog_np, skt_np, labels_np, subject_np, test_indices)

print('\nSingle fixed subject-dependent split:')
print('train size:', len(train_dataset))
print('test size :', len(test_dataset))
print('train label counts:', np.bincount(train_dataset.labels.numpy()))
print('test label counts :', np.bincount(test_dataset.labels.numpy()))
if split_notes:
    print('\nSplit notes:')
    for note in split_notes:
        print('-', note)
else:
    print('\nAll per-subject splits used stratification successfully.')


# Model Definition

The encoders, fusion heads, and classifier below are written directly in the notebook, but they intentionally follow the same style as the baseline `MFMC_Fusion_MLP` notebook:

- each modality is encoded into a 128-dimensional representation,
- each pair-fusion head is `256 -> 512 -> 128` with `BatchNorm + ReLU`, and
- the downstream classifier is a light MLP over the EEG embedding.


In [ ]:
class NETWORK_F_MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=200, out_dim=200, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        self.fc_list = nn.ModuleList()
        self.bn_list = nn.ModuleList()
        self.fc_list.append(nn.Linear(input_dim, hidden_dim, bias=True))
        self.bn_list.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(self.num_layers - 1):
            self.fc_list.append(nn.Linear(hidden_dim, hidden_dim, bias=True))
            self.bn_list.append(nn.BatchNorm1d(hidden_dim))
        self.fc_final = nn.Linear(hidden_dim, out_dim, bias=True)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)
        for i in range(self.num_layers):
            x = self.fc_list[i](x)
            x = torch.relu(x)
            x = self.bn_list[i](x)
        x = self.fc_final(x)
        x = torch.sigmoid(x)
        return x


class Advanced1DCNNChannel(nn.Module):
    def __init__(self, input_channels=1, embed_dim=128, input_size=1280):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=11, padding=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv4 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=11, padding=5),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        feat_size = input_size // (4 * 4 * 4 * 4)
        self.fc1 = nn.Sequential(
            nn.Linear(256 * feat_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
        )
        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
        )
        self.fc3 = nn.Linear(512, embed_dim)
        self.mlp = NETWORK_F_MLP(
            input_dim=embed_dim * input_channels,
            hidden_dim=4000,
            out_dim=embed_dim,
            num_layers=1,
        )

    def forward(self, x):
        batch_size, channels = x.shape[0], x.shape[1]
        x = x.unsqueeze(2)
        x = x.flatten(0, 1)
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)
        out = out.reshape(batch_size, channels, -1)
        out = out.flatten(-2, -1)
        out = self.mlp(out)
        return out


class PairFusionHead(nn.Module):
    def __init__(self, input_dim=256, hidden_dim=512, output_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

    def forward(self, left, right):
        x = torch.cat([left, right], dim=1)
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        return x


class EEGClassifier(nn.Module):
    def __init__(self, embed_dim=128, num_classes=4):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x


def build_models(device):
    eeg_encoder = Advanced1DCNNChannel(
        input_channels=eeg_np.shape[1],
        embed_dim=EMBED_DIM,
        input_size=eeg_np.shape[2],
    ).to(device)
    eog_encoder = Advanced1DCNNChannel(
        input_channels=eog_np.shape[1],
        embed_dim=EMBED_DIM,
        input_size=eog_np.shape[2],
    ).to(device)
    skt_encoder = Advanced1DCNNChannel(
        input_channels=skt_np.shape[1],
        embed_dim=EMBED_DIM,
        input_size=skt_np.shape[2],
    ).to(device)
    fusion_12 = PairFusionHead(EMBED_DIM * 2, 512, EMBED_DIM).to(device)
    fusion_13 = PairFusionHead(EMBED_DIM * 2, 512, EMBED_DIM).to(device)
    fusion_23 = PairFusionHead(EMBED_DIM * 2, 512, EMBED_DIM).to(device)
    classifier = EEGClassifier(EMBED_DIM, int(labels_np.max()) + 1).to(device)
    return {
        'eeg_encoder': eeg_encoder,
        'eog_encoder': eog_encoder,
        'skt_encoder': skt_encoder,
        'fusion_12': fusion_12,
        'fusion_13': fusion_13,
        'fusion_23': fusion_23,
        'classifier': classifier,
    }


# Trace-based MFMC Loss with Ridge Regularization

The implementation below replaces explicit matrix inversion with `torch.linalg.solve`, wraps each `ε` run in its own exception-safe block, and records conditioning statistics for the stabilized matrices:

- `cond(R1_tilde)`, `cond(R2_tilde)`, `cond(R3_tilde)`
- `cond(R12_tilde)`, `cond(R13_tilde)`, `cond(R23_tilde)`

The tri-modal cyclic terms follow the reviewer-requested structure:

- `term_12_3 = trace(R12^{-1} P12,3 R3^{-1} P12,3^T)`
- `term_13_2 = trace(R13^{-1} P13,2 R2^{-1} P13,2^T)`
- `term_23_1 = trace(R23^{-1} P23,1 R1^{-1} P23,1^T)`

The optimization loss is the negative sum of those three scores so that maximizing dependence becomes a minimization problem.


In [ ]:
@dataclass
class FMCAComputation:
    loss: torch.Tensor
    total_score: torch.Tensor
    terms: dict
    conds: dict
    nan_or_inf: bool
    notes: list


class LinearSolveFailure(RuntimeError):
    pass


class DivergenceError(RuntimeError):
    pass



def make_loader(dataset, batch_size, shuffle, drop_last, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=0,
        generator=generator,
    )



def cycle_loader(loader):
    while True:
        for batch in loader:
            yield batch



def batch_to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}



def adaptive_estimation(v_t, beta, square_term, i):
    v_t = beta * v_t + (1 - beta) * square_term.detach()
    return v_t, (v_t / (1 - beta ** i))



def ridge_regularize(R, eps):
    eye = torch.eye(R.shape[0], device=R.device, dtype=R.dtype)
    scale = torch.trace(R) / R.shape[0]
    return R + eps * scale * eye



def safe_condition_number(matrix):
    try:
        cond_value = torch.linalg.cond(matrix)
        value = float(cond_value.detach().cpu())
        if not np.isfinite(value):
            return float('inf')
        return value
    except Exception:
        return float('inf')



def cross_autocorrelation(left, right):
    B = left.shape[0]
    R_left = (left.T @ left) / B
    R_right = (right.T @ right) / B
    P = (left.T @ right) / B
    return R_left, R_right, P



def init_covariance_trackers(device, feature_dim=EMBED_DIM):
    return {
        key: {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
        }
        for key in ['track_12_3', 'track_13_2', 'track_23_1']
    }



def mfmc_t_trace(zx, zy, track_cov, step, eps, cov_beta=COV_BETA, prefix='fmca'):
    zx64 = zx.to(LOSS_DTYPE)
    zy64 = zy.to(LOSS_DTYPE)
    Rx, Ry, Pxy = cross_autocorrelation(zx64, zy64)
    Rx = ridge_regularize(Rx, eps)
    Ry = ridge_regularize(Ry, eps)

    conds = {
        f'cond_{prefix}_left': safe_condition_number(Rx),
        f'cond_{prefix}_right': safe_condition_number(Ry),
    }

    track_cov['Rx'], Rx_est = adaptive_estimation(track_cov['Rx'], cov_beta, Rx, step)
    track_cov['Ry'], Ry_est = adaptive_estimation(track_cov['Ry'], cov_beta, Ry, step)
    track_cov['Pxy'], Pxy_est = adaptive_estimation(track_cov['Pxy'], cov_beta, Pxy, step)

    try:
        Rx_est_inv = torch.inverse(Rx_est)
        Ry_est_inv = torch.inverse(Ry_est)
    except RuntimeError as exc:
        raise LinearSolveFailure(f'Matrix inverse failed for {prefix} at eps={eps}: {exc}') from exc

    cost = (
        -Rx_est_inv @ Rx @ Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy_est.T
        + Rx_est_inv @ Pxy @ Ry_est_inv @ Pxy_est.T
        - Rx_est_inv @ Pxy_est @ Ry_est_inv @ Ry @ Ry_est_inv @ Pxy_est.T
        + Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy.T
    )
    loss = -torch.trace(cost)

    if not torch.isfinite(loss):
        raise DivergenceError(f'Non-finite MFMC loss encountered for {prefix} at eps={eps}.')

    return track_cov, loss.to(zx.dtype), conds, float((-loss).detach().cpu())



def mfmc_tri_modal_loss(e1, e2, e3, e12, e13, e23, trackers, step, eps, cov_beta=COV_BETA):
    trackers['track_12_3'], loss_12_3, conds_12_3, term_12_3 = mfmc_t_trace(
        e12, e3, trackers['track_12_3'], step, eps, cov_beta, 'R12_R3'
    )
    trackers['track_13_2'], loss_13_2, conds_13_2, term_13_2 = mfmc_t_trace(
        e13, e2, trackers['track_13_2'], step, eps, cov_beta, 'R13_R2'
    )
    trackers['track_23_1'], loss_23_1, conds_23_1, term_23_1 = mfmc_t_trace(
        e23, e1, trackers['track_23_1'], step, eps, cov_beta, 'R23_R1'
    )

    all_conds = {}
    all_conds.update(conds_12_3)
    all_conds.update(conds_13_2)
    all_conds.update(conds_23_1)

    total_loss = loss_12_3 + loss_13_2 + loss_23_1
    total_score = -total_loss
    nan_or_inf = (not torch.isfinite(total_loss)) or any(not np.isfinite(v) for v in all_conds.values())

    if abs(float(total_loss.detach().cpu())) > MAX_REASONABLE_LOSS:
        raise DivergenceError(
            f'Loss magnitude exceeded MAX_REASONABLE_LOSS={MAX_REASONABLE_LOSS:.1e}. Observed {float(total_loss.detach().cpu()):.4e}.'
        )

    return trackers, FMCAComputation(
        loss=total_loss,
        total_score=total_score,
        terms={
            'term_12_3': term_12_3,
            'term_13_2': term_13_2,
            'term_23_1': term_23_1,
        },
        conds=all_conds,
        nan_or_inf=nan_or_inf,
        notes=[],
    )



def encode_modalities(batch, models):
    e1 = models['eeg_encoder'](batch['eeg'])
    e2 = models['eog_encoder'](batch['eog'])
    e3 = models['skt_encoder'](batch['skt'])
    e12 = models['fusion_12'](e1, e2)
    e13 = models['fusion_13'](e1, e3)
    e23 = models['fusion_23'](e2, e3)
    return e1, e2, e3, e12, e13, e23



def evaluate_classifier(models, loader, device):
    models['eeg_encoder'].eval()
    models['classifier'].eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for batch in loader:
            batch = batch_to_device(batch, device)
            features = models['eeg_encoder'](batch['eeg'])
            logits = models['classifier'](features)
            preds = torch.argmax(logits, dim=1)
            y_true.extend(batch['label'].cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    models['eeg_encoder'].train()
    models['classifier'].train()
    return acc, macro_f1



def collect_condition_snapshot(models, ref_batch, eps):
    with torch.no_grad():
        e1, e2, e3, e12, e13, e23 = encode_modalities(ref_batch, models)
        pair_map = {
            'R12_R3': (e12.to(LOSS_DTYPE), e3.to(LOSS_DTYPE)),
            'R13_R2': (e13.to(LOSS_DTYPE), e2.to(LOSS_DTYPE)),
            'R23_R1': (e23.to(LOSS_DTYPE), e1.to(LOSS_DTYPE)),
        }
        condensed = {}
        for prefix, (left, right) in pair_map.items():
            Rx, Ry, _ = cross_autocorrelation(left, right)
            condensed[f'cond_{prefix}_left'] = safe_condition_number(ridge_regularize(Rx, eps))
            condensed[f'cond_{prefix}_right'] = safe_condition_number(ridge_regularize(Ry, eps))
    return condensed



# Single-split Training and Evaluation

The training loop follows the baseline notebook's alternating structure:

1. one MFMC self-supervised update for the three encoders and three fusion heads,
2. one supervised cross-entropy update for the classifier using the EEG encoder representation, and
3. periodic evaluation on the fixed held-out split.

Before the full `ε` sweep, the next cell runs a short smoke test only. It checks imports, data loading, one mini-batch forward pass, the ridge-stabilized FMCA loss at `ε = 1e-4`, `backward()`, and one classifier evaluation call. It does **not** launch the full 10,000-iteration sweep.


In [ ]:
criterion = nn.CrossEntropyLoss()


def build_data_loaders(seed):
    train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True, drop_last=True, seed=seed)
    test_loader = make_loader(test_dataset, CLASSIFIER_EVAL_BATCH_SIZE, shuffle=False, drop_last=False, seed=seed)
    return train_loader, test_loader



def run_smoke_test(seed=RANDOM_SEED, eps=1e-4):
    set_seed(seed)
    models = build_models(device)
    train_loader, test_loader = build_data_loaders(seed)
    feature_params = (
        list(models['eeg_encoder'].parameters())
        + list(models['eog_encoder'].parameters())
        + list(models['skt_encoder'].parameters())
        + list(models['fusion_12'].parameters())
        + list(models['fusion_13'].parameters())
        + list(models['fusion_23'].parameters())
    )
    optimizer_features = optim.Adam(feature_params, lr=LR, betas=BETAS, amsgrad=True)
    optimizer_classifier = optim.Adam(models['classifier'].parameters(), lr=LR, betas=BETAS, amsgrad=True)
    trackers = init_covariance_trackers(device)

    batch = next(iter(train_loader))
    batch = batch_to_device(batch, device)

    optimizer_features.zero_grad(set_to_none=True)
    e1, e2, e3, e12, e13, e23 = encode_modalities(batch, models)
    trackers, comp = mfmc_tri_modal_loss(e1, e2, e3, e12, e13, e23, trackers, step=1, eps=eps, cov_beta=COV_BETA)
    comp.loss.backward()
    optimizer_features.step()

    optimizer_classifier.zero_grad(set_to_none=True)
    with torch.no_grad():
        eeg_features = models['eeg_encoder'](batch['eeg'])
    logits = models['classifier'](eeg_features.detach())
    classifier_loss = criterion(logits, batch['label'])
    classifier_loss.backward()
    optimizer_classifier.step()

    acc, macro_f1 = evaluate_classifier(models, test_loader, device)

    results = {
        'smoke_test_passed': True,
        'device': str(device),
        'batch_size': int(batch['label'].shape[0]),
        'eeg_embedding_shape': tuple(e1.shape),
        'eog_embedding_shape': tuple(e2.shape),
        'skt_embedding_shape': tuple(e3.shape),
        'e12_shape': tuple(e12.shape),
        'e13_shape': tuple(e13.shape),
        'e23_shape': tuple(e23.shape),
        'mfmc_loss_eps_1e4': float(comp.loss.detach().cpu()),
        'classifier_loss': float(classifier_loss.detach().cpu()),
        'smoke_accuracy': float(acc),
        'smoke_macro_f1': float(macro_f1),
        'nan_or_inf_detected': bool(comp.nan_or_inf),
    }

    del models, optimizer_features, optimizer_classifier, batch, e1, e2, e3, e12, e13, e23, logits
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


smoke_test_results = run_smoke_test()
display(pd.DataFrame([smoke_test_results]))


# Epsilon Sensitivity Results

The cell below launches the full single-split sweep over:

`EPS_LIST = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]`

Every run reuses the same train/test split, the same batch size, the same optimizer configuration, and the same iteration budget. The only varying factor is the ridge coefficient `ε`.

For multi-GPU execution, the notebook assigns independent `ε` runs to separate CUDA devices. This is task-level parallelism rather than model-level data parallelism: each GPU owns one complete encoder/fusion/classifier stack for one epsilon value.

To keep notebook output readable, worker runs do not print training logs. They send structured progress records to the notebook kernel, and the main thread refreshes a single dashboard table with `clear_output(wait=True)`. Optional disk output is limited to fixed filenames under `_epsilon_multigpu_outputs/`, overwritten on each sweep:

- `latest_summary.csv`
- `latest_results.pkl`

Set `SAVE_MULTIGPU_RESULTS = False` if you want purely in-memory notebook outputs.


In [ ]:
import pickle
import queue
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from IPython.display import clear_output

RUN_FULL_EPSILON_SWEEP = True
USE_MULTIGPU = True
PREFERRED_GPU_IDS = list(range(6))
PROGRESS_REFRESH_SECONDS = 2.0
SAVE_MULTIGPU_RESULTS = True
MULTIGPU_OUTPUT_DIR = PROJECT_ROOT / 'Supplement' / 'epsilon_sensitivity_trace_fmca' / '_epsilon_multigpu_outputs'
SUMMARY_CSV_PATH = MULTIGPU_OUTPUT_DIR / 'latest_summary.csv'
RESULTS_PKL_PATH = MULTIGPU_OUTPUT_DIR / 'latest_results.pkl'
MODEL_INIT_LOCK = threading.Lock()


def summarize_condition_numbers(cond_history):
    values = []
    for record in cond_history:
        values.extend(list(record['conds'].values()))
    if not values:
        return np.nan, np.nan, np.nan
    arr = np.asarray(values, dtype=float)
    cond_mean = float(np.mean(arr)) if np.all(np.isfinite(arr)) else float('inf')
    cond_median = float(np.median(arr)) if np.all(np.isfinite(arr)) else float('inf')
    cond_max = float(np.max(arr))
    return cond_mean, cond_median, cond_max



def send_progress(progress_queue, **payload):
    if progress_queue is not None:
        progress_queue.put(payload)



def run_single_epsilon(eps, seed=RANDOM_SEED, run_device=None, gpu_id=None, progress_queue=None):
    run_device = run_device if run_device is not None else device
    if run_device.type == 'cuda':
        torch.cuda.set_device(run_device)

    send_progress(
        progress_queue,
        epsilon=eps,
        gpu_id=gpu_id,
        status='initializing',
        iteration=0,
        latest_acc=np.nan,
        latest_macro_f1=np.nan,
        latest_loss=np.nan,
        cond_max=np.nan,
        note='',
    )

    with MODEL_INIT_LOCK:
        set_seed(seed)
        models = build_models(run_device)

    train_loader, test_loader = build_data_loaders(seed)
    train_iter = cycle_loader(train_loader)
    ref_batch = batch_to_device(
        next(iter(make_loader(train_dataset, BATCH_SIZE, shuffle=False, drop_last=True, seed=seed))),
        run_device,
    )

    feature_params = (
        list(models['eeg_encoder'].parameters())
        + list(models['eog_encoder'].parameters())
        + list(models['skt_encoder'].parameters())
        + list(models['fusion_12'].parameters())
        + list(models['fusion_13'].parameters())
        + list(models['fusion_23'].parameters())
    )
    optimizer_features = optim.Adam(feature_params, lr=LR, betas=BETAS, amsgrad=True)
    optimizer_classifier = optim.Adam(models['classifier'].parameters(), lr=LR, betas=BETAS, amsgrad=True)
    trackers = init_covariance_trackers(run_device)

    history = {
        'iteration': [],
        'accuracy': [],
        'macro_f1': [],
        'cond_mean': [],
        'cond_max': [],
        'mfmc_loss': [],
    }
    cond_history = []
    notes = []
    nan_or_inf_detected = False
    best_acc = -np.inf
    best_f1 = -np.inf
    final_acc = np.nan
    final_f1 = np.nan
    status = 'completed'

    try:
        send_progress(
            progress_queue,
            epsilon=eps,
            gpu_id=gpu_id,
            status='running',
            iteration=0,
            latest_acc=np.nan,
            latest_macro_f1=np.nan,
            latest_loss=np.nan,
            cond_max=np.nan,
            note='',
        )

        for iteration in range(1, N_ITERS + 1):
            unsup_batch = batch_to_device(next(train_iter), run_device)
            optimizer_features.zero_grad(set_to_none=True)
            e1, e2, e3, e12, e13, e23 = encode_modalities(unsup_batch, models)
            trackers, fmca_comp = mfmc_tri_modal_loss(
                e1, e2, e3, e12, e13, e23, trackers, step=iteration, eps=eps, cov_beta=COV_BETA
            )
            nan_or_inf_detected = nan_or_inf_detected or fmca_comp.nan_or_inf or (not torch.isfinite(fmca_comp.loss))
            if not torch.isfinite(fmca_comp.loss):
                raise DivergenceError(f'Non-finite MFMC loss at iteration {iteration}.')
            fmca_comp.loss.backward()
            optimizer_features.step()

            sup_batch = batch_to_device(next(train_iter), run_device)
            optimizer_classifier.zero_grad(set_to_none=True)
            with torch.no_grad():
                eeg_features = models['eeg_encoder'](sup_batch['eeg'])
            logits = models['classifier'](eeg_features.detach())
            classifier_loss = criterion(logits, sup_batch['label'])
            if not torch.isfinite(classifier_loss):
                nan_or_inf_detected = True
                raise DivergenceError(f'Non-finite classifier loss at iteration {iteration}.')
            classifier_loss.backward()
            optimizer_classifier.step()

            if iteration % EVAL_EVERY == 0:
                acc, macro_f1 = evaluate_classifier(models, test_loader, run_device)
                cond_snapshot = collect_condition_snapshot(models, ref_batch, eps)
                cond_values = np.asarray(list(cond_snapshot.values()), dtype=float)
                cond_mean = float(np.mean(cond_values)) if np.all(np.isfinite(cond_values)) else float('inf')
                cond_max = float(np.max(cond_values))
                latest_loss = float(fmca_comp.loss.detach().cpu())

                history['iteration'].append(iteration)
                history['accuracy'].append(float(acc))
                history['macro_f1'].append(float(macro_f1))
                history['cond_mean'].append(cond_mean)
                history['cond_max'].append(cond_max)
                history['mfmc_loss'].append(latest_loss)
                cond_history.append({'iteration': iteration, 'conds': cond_snapshot})

                best_acc = max(best_acc, float(acc))
                best_f1 = max(best_f1, float(macro_f1))
                final_acc = float(acc)
                final_f1 = float(macro_f1)

                note = ''
                if cond_max > 1e12:
                    note = f'Condition number exceeded 1e12 at iteration {iteration}.'
                    notes.append(note)

                send_progress(
                    progress_queue,
                    epsilon=eps,
                    gpu_id=gpu_id,
                    status='running',
                    iteration=iteration,
                    latest_acc=float(acc),
                    latest_macro_f1=float(macro_f1),
                    best_acc=best_acc,
                    best_macro_f1=best_f1,
                    latest_loss=latest_loss,
                    cond_max=cond_max,
                    note=note,
                )

    except LinearSolveFailure as exc:
        status = 'failed'
        nan_or_inf_detected = True
        notes.append(str(exc))
    except DivergenceError as exc:
        status = 'diverged'
        nan_or_inf_detected = True
        notes.append(str(exc))
    except RuntimeError as exc:
        status = 'failed'
        nan_or_inf_detected = True
        notes.append(f'RuntimeError: {exc}')
    except Exception as exc:
        status = 'failed'
        nan_or_inf_detected = True
        notes.append(f'Unexpected error: {exc}')

    cond_mean, cond_median, cond_max = summarize_condition_numbers(cond_history)
    if status != 'completed' and not notes:
        notes.append('Run did not complete, but no explicit note was captured.')

    summary = {
        'epsilon': eps,
        'status': status,
        'best_acc': np.nan if best_acc == -np.inf else best_acc,
        'final_acc': final_acc,
        'best_macro_f1': np.nan if best_f1 == -np.inf else best_f1,
        'final_macro_f1': final_f1,
        'cond_mean': cond_mean,
        'cond_median': cond_median,
        'cond_max': cond_max,
        'nan_or_inf_detected': bool(nan_or_inf_detected),
        'notes': ' | '.join(dict.fromkeys(notes)) if notes else '',
    }

    send_progress(
        progress_queue,
        epsilon=eps,
        gpu_id=gpu_id,
        status=status,
        iteration=history['iteration'][-1] if history['iteration'] else 0,
        latest_acc=final_acc,
        latest_macro_f1=final_f1,
        best_acc=summary['best_acc'],
        best_macro_f1=summary['best_macro_f1'],
        latest_loss=history['mfmc_loss'][-1] if history['mfmc_loss'] else np.nan,
        cond_max=summary['cond_max'],
        note=summary['notes'],
        done=True,
    )

    del models, optimizer_features, optimizer_classifier, train_loader, test_loader, train_iter, ref_batch
    gc.collect()
    if run_device.type == 'cuda':
        torch.cuda.empty_cache()

    return summary, history, cond_history



def format_epsilon(eps):
    return f'{eps:.0e}' if eps != 0 else '0'



def available_gpu_ids(preferred_gpu_ids=None):
    if not torch.cuda.is_available():
        return []
    count = torch.cuda.device_count()
    if preferred_gpu_ids is None:
        return list(range(count))
    return [gpu_id for gpu_id in preferred_gpu_ids if 0 <= gpu_id < count]



def build_gpu_plan(eps_list, preferred_gpu_ids=None):
    gpu_ids = available_gpu_ids(preferred_gpu_ids)
    if USE_MULTIGPU and gpu_ids:
        assignments = {eps: gpu_ids[idx % len(gpu_ids)] for idx, eps in enumerate(eps_list)}
        max_workers = min(len(eps_list), len(gpu_ids))
        return assignments, max_workers
    return {eps: None for eps in eps_list}, 1



def initial_progress_state(eps_gpu_map):
    state = {}
    for eps, gpu_id in eps_gpu_map.items():
        state[eps] = {
            'epsilon': eps,
            'gpu': 'cpu' if gpu_id is None else f'cuda:{gpu_id}',
            'status': 'queued',
            'iteration': 0,
            'progress': 0.0,
            'latest_acc': np.nan,
            'latest_macro_f1': np.nan,
            'best_acc': np.nan,
            'best_macro_f1': np.nan,
            'latest_loss': np.nan,
            'cond_max': np.nan,
            'note': '',
        }
    return state



def apply_progress_message(progress_state, message):
    eps = message['epsilon']
    row = progress_state[eps]
    row['status'] = message.get('status', row['status'])
    row['iteration'] = int(message.get('iteration', row['iteration']) or 0)
    row['progress'] = min(row['iteration'] / max(N_ITERS, 1), 1.0)
    for key in ['latest_acc', 'latest_macro_f1', 'best_acc', 'best_macro_f1', 'latest_loss', 'cond_max', 'note']:
        if key in message:
            row[key] = message[key]
    if message.get('gpu_id') is not None:
        row['gpu'] = f"cuda:{message['gpu_id']}"



def render_progress_dashboard(progress_state, completed_summaries=None, force=False):
    dashboard_df = pd.DataFrame([progress_state[eps] for eps in EPS_LIST])
    dashboard_df['epsilon'] = dashboard_df['epsilon'].map(format_epsilon)
    dashboard_df['progress'] = (dashboard_df['progress'] * 100).map(lambda x: f'{x:5.1f}%')
    for col in ['latest_acc', 'latest_macro_f1', 'best_acc', 'best_macro_f1']:
        dashboard_df[col] = dashboard_df[col].map(lambda x: '' if pd.isna(x) else f'{x:.4f}')
    dashboard_df['latest_loss'] = dashboard_df['latest_loss'].map(lambda x: '' if pd.isna(x) else f'{x:.4g}')
    dashboard_df['cond_max'] = dashboard_df['cond_max'].map(lambda x: '' if pd.isna(x) else f'{x:.3e}')

    clear_output(wait=True)
    display(Markdown('### Multi-GPU epsilon sweep progress'))
    display(dashboard_df[['epsilon', 'gpu', 'status', 'iteration', 'progress', 'latest_acc', 'latest_macro_f1', 'best_acc', 'best_macro_f1', 'latest_loss', 'cond_max', 'note']])
    if completed_summaries:
        display(Markdown(f'Completed runs: **{len(completed_summaries)} / {len(EPS_LIST)}**'))



def epsilon_worker(eps, gpu_id, progress_queue):
    run_device = torch.device('cpu') if gpu_id is None else torch.device(f'cuda:{gpu_id}')
    return run_single_epsilon(
        eps,
        seed=RANDOM_SEED,
        run_device=run_device,
        gpu_id=gpu_id,
        progress_queue=progress_queue,
    )



def run_epsilon_sweep_multigpu():
    eps_gpu_map, max_workers = build_gpu_plan(EPS_LIST, PREFERRED_GPU_IDS)
    progress_queue = queue.Queue()
    progress_state = initial_progress_state(eps_gpu_map)
    completed_summaries = []
    all_histories = {}
    all_cond_histories = {}

    render_progress_dashboard(progress_state, completed_summaries, force=True)
    start_time = time.time()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_eps = {
            executor.submit(epsilon_worker, eps, gpu_id, progress_queue): eps
            for eps, gpu_id in eps_gpu_map.items()
        }
        unfinished = set(future_to_eps)
        last_render = 0.0

        while unfinished:
            changed = False
            while True:
                try:
                    message = progress_queue.get_nowait()
                except queue.Empty:
                    break
                apply_progress_message(progress_state, message)
                changed = True

            for future in list(unfinished):
                if future.done():
                    eps = future_to_eps[future]
                    try:
                        summary, history, cond_history = future.result()
                    except Exception as exc:
                        summary = {
                            'epsilon': eps,
                            'status': 'failed',
                            'best_acc': np.nan,
                            'final_acc': np.nan,
                            'best_macro_f1': np.nan,
                            'final_macro_f1': np.nan,
                            'cond_mean': np.nan,
                            'cond_median': np.nan,
                            'cond_max': np.nan,
                            'nan_or_inf_detected': True,
                            'notes': f'Worker future failed: {exc}',
                        }
                        history = {'iteration': [], 'accuracy': [], 'macro_f1': [], 'cond_mean': [], 'cond_max': [], 'mfmc_loss': []}
                        cond_history = []
                    completed_summaries.append(summary)
                    all_histories[eps] = history
                    all_cond_histories[eps] = cond_history
                    unfinished.remove(future)
                    changed = True

            now = time.time()
            if changed or now - last_render >= PROGRESS_REFRESH_SECONDS:
                render_progress_dashboard(progress_state, completed_summaries)
                last_render = now
            time.sleep(0.25)

    elapsed = time.time() - start_time
    summary_df = pd.DataFrame(completed_summaries).sort_values('epsilon').reset_index(drop=True)
    render_progress_dashboard(progress_state, completed_summaries, force=True)
    display(Markdown(f'Full epsilon sweep finished in **{elapsed / 60:.1f} minutes** using **{max_workers}** worker(s).'))
    return summary_df, all_histories, all_cond_histories, eps_gpu_map



def save_limited_results(summary_df, histories, cond_histories, eps_gpu_map):
    if not SAVE_MULTIGPU_RESULTS:
        return
    MULTIGPU_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
    with open(RESULTS_PKL_PATH, 'wb') as f:
        pickle.dump(
            {
                'summary_df': summary_df,
                'histories': histories,
                'cond_histories': cond_histories,
                'eps_gpu_map': eps_gpu_map,
                'config': {
                    'random_seed': RANDOM_SEED,
                    'batch_size': BATCH_SIZE,
                    'n_iters': N_ITERS,
                    'eval_every': EVAL_EVERY,
                    'lr': LR,
                    'betas': BETAS,
                    'embed_dim': EMBED_DIM,
                    'eps_list': EPS_LIST,
                },
            },
            f,
        )
    display(Markdown(f'Limited result files overwritten in `{MULTIGPU_OUTPUT_DIR}`.'))
    display(pd.DataFrame([{'file': str(SUMMARY_CSV_PATH)}, {'file': str(RESULTS_PKL_PATH)}]))



def plot_histories(summary_df, histories):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for eps in EPS_LIST:
        hist = histories.get(eps, {})
        if not hist.get('iteration'):
            continue
        label = f'ε={format_epsilon(eps)}'
        status = summary_df.loc[summary_df['epsilon'] == eps, 'status'].iloc[0]
        linestyle = '--' if status != 'completed' else '-'
        axes[0].plot(hist['iteration'], hist['accuracy'], label=label, linestyle=linestyle)
        axes[1].plot(hist['iteration'], hist['macro_f1'], label=label, linestyle=linestyle)
        axes[2].plot(hist['iteration'], hist['cond_max'], label=label, linestyle=linestyle)

    axes[0].set_title('Accuracy vs Training Iteration')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Accuracy')
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title('Macro-F1 vs Training Iteration')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Macro-F1')
    axes[1].grid(True, alpha=0.3)

    axes[2].set_title('Condition Number Summary')
    axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('Max condition number across tracked matrices')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper center', ncol=min(len(handles), 6), bbox_to_anchor=(0.5, 1.05))
    plt.tight_layout()
    plt.show()

    bar_df = summary_df.copy()
    bar_df['epsilon_label'] = bar_df['epsilon'].map(format_epsilon)
    finite_mask = np.isfinite(bar_df['cond_max'])
    plt.figure(figsize=(8, 4))
    plt.bar(bar_df.loc[finite_mask, 'epsilon_label'], bar_df.loc[finite_mask, 'cond_max'])
    plt.yscale('log')
    plt.title('cond_max by epsilon')
    plt.xlabel('epsilon')
    plt.ylabel('cond_max (log scale)')
    plt.grid(True, axis='y', alpha=0.3)
    plt.show()



def build_reviewer_summary(summary_df):
    completed = summary_df[summary_df['status'] == 'completed'].copy().sort_values('epsilon')
    failed = summary_df[summary_df['status'] != 'completed'].copy().sort_values('epsilon')

    if completed.empty:
        failed_eps = ', '.join(format_epsilon(v) for v in failed['epsilon'].tolist()) or 'none'
        return (
            'No epsilon setting completed successfully in this diagnostic run. '
            f'Failed or diverged settings: {failed_eps}. '
            'This indicates that the current split or optimization configuration should be inspected before drawing a reviewer-facing conclusion.'
        )

    best_acc = completed['best_acc'].max()
    best_f1 = completed['best_macro_f1'].max()
    comparable = completed[
        (completed['best_acc'] >= best_acc - 0.02)
        & (completed['best_macro_f1'] >= best_f1 - 0.02)
    ]

    completed_eps = completed['epsilon'].tolist()
    comparable_eps = comparable['epsilon'].tolist() if not comparable.empty else completed_eps
    stable_range = f"[{format_epsilon(min(completed_eps))}, {format_epsilon(max(completed_eps))}]"
    comparable_range = f"[{format_epsilon(min(comparable_eps))}, {format_epsilon(max(comparable_eps))}]"

    failed_text = 'none'
    if not failed.empty:
        failed_text = ', '.join(
            f"{format_epsilon(row.epsilon)} ({row.status})" for row in failed.itertuples()
        )

    default_row = summary_df.loc[summary_df['epsilon'] == 1e-4].iloc[0]
    default_text = (
        f"At ε=1e-4, best accuracy = {default_row['best_acc']:.4f}, "
        f"best macro-F1 = {default_row['best_macro_f1']:.4f}, "
        f"and cond_max = {default_row['cond_max']:.4e}."
        if np.isfinite(default_row['best_acc'])
        else 'The ε=1e-4 run did not complete successfully in this execution.'
    )

    zero_row = summary_df.loc[summary_df['epsilon'] == 0.0].iloc[0]
    zero_text = ''
    if np.isfinite(zero_row['cond_max']) and np.isfinite(default_row['cond_max']):
        improvement = zero_row['cond_max'] / max(default_row['cond_max'], 1e-12)
        zero_text = f' Relative to ε=0, the cond_max ratio was approximately {improvement:.2e}.'
    elif zero_row['status'] != 'completed':
        zero_text = ' The ε=0 run did not complete cleanly, which further supports the need for ridge stabilization.'

    return (
        f"The trace-based FMCA objective completed successfully for ε in {stable_range}. "
        f"Across the completed runs, the downstream metrics were most comparable in {comparable_range}. "
        f"Failed or diverged settings: {failed_text}. "
        + default_text
        + zero_text
        + ' In this notebook, ε=1e-4 is the default recommendation because it is intended to improve conditioning while perturbing the autocorrelation structure only mildly; the exact strength of that recommendation should be judged from the numbers above.'
    )


if RUN_FULL_EPSILON_SWEEP:
    summary_df, all_histories, all_cond_histories, eps_gpu_map = run_epsilon_sweep_multigpu()
    display(summary_df)
    save_limited_results(summary_df, all_histories, all_cond_histories, eps_gpu_map)
    plot_histories(summary_df, all_histories)
    reviewer_summary_text = build_reviewer_summary(summary_df)
else:
    summary_df = pd.DataFrame(columns=['epsilon', 'status', 'best_acc', 'final_acc', 'best_macro_f1', 'final_macro_f1', 'cond_mean', 'cond_median', 'cond_max', 'nan_or_inf_detected', 'notes'])
    all_histories = {}
    all_cond_histories = {}
    eps_gpu_map = {}
    reviewer_summary_text = 'Full epsilon sweep has not been run. Set RUN_FULL_EPSILON_SWEEP = True and execute this cell to generate the reviewer-response summary.'
    display(Markdown(reviewer_summary_text))


# Reviewer-response Summary

The conclusion below is generated automatically from the actual run results in `summary_df`, rather than being hard-coded in advance.


In [ ]:
print(reviewer_summary_text)
display(Markdown(f"**Reviewer-response Summary**\n\n{reviewer_summary_text}"))
